# 2. PostgreSQL para MongoDB

**Equipe:** Diogo Galrão Carvalho; Felipe Artur Macedo Lima; Luan Cavalcante Dias Rodrigues.

**Disciplina:** Banco de Dados e Big Data para Data Science, Pós-Graduação em Data Science e Analytics.

**Ambiente executado: Docker local.** Não houve execução em Atlas. O roteiro pede Atlas; a substituição pelo MongoDB local deve ser alinhada com o docente.

In [1]:
from pathlib import Path
import sys
import os
ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import json
import pandas as pd
from scripts.migrate_to_mongodb import load_tables, build_documents, migrate
tables = load_tables("postgres")
csv_tables = load_tables("csv")
assert build_documents(tables) == build_documents(csv_tables)
display(pd.DataFrame([{"tabela_postgresql": k, "linhas": len(v)} for k, v in tables.items()]))

,tabela_postgresql,linhas
0,elo,10
1,categoria,32
2,item,254
3,item_categoria,834
4,jogador,25
5,partida,30
6,compra,1043


## Escolhas de modelagem

- `jogadores`: elo textual e compras embutidos; `id_elo` preservado.
- `itens`: catálogo separado; categorias com ID/nome embutidas.
- `partidas`: coleção referenciada por `id_partida`.
- `dominios`: um documento conserva todos os elos e categorias, inclusive os não utilizados.

O preço vem da transação original; o nome do item vem do catálogo no momento da migração. A origem não guarda versões dos nomes. Como o array de compras não tem limite lógico, seu tamanho BSON precisa ser acompanhado e a modelagem revista antes de atingir 16 MiB.

In [2]:
result = migrate(source="postgres")
print(json.dumps({k: v for k, v in result.items() if k != "sample"}, ensure_ascii=False, indent=2))

{
  "executed_at": "2026-09-29T00:05:02.650134+00:00",
  "source": "postgres",
  "source_counts": {
    "elo": 10,
    "categoria": 32,
    "item": 254,
    "item_categoria": 834,
    "jogador": 25,
    "partida": 30,
    "compra": 1043
  },
  "input_sha256": {
    "data/processed/categoria.csv": "015dd5ba54c07ff22abdde9c3b7793d58dbb088afb267b519a971b832644d6e3",
    "data/processed/compra.csv": "7571f4a1c0300a461341e17ddadca05d79a4582e10b3c846d378afe90263ee5c",
    "data/processed/elo.csv": "1993fd793488dbde1247c3e1d06d349ab80b99b2def1242bbcd307117de01a1f",
    "data/processed/item.csv": "d37617b03ae33d5027573a7bfb630b5c4d97d9c7d4226f80a7b79025d29be489",
    "data/processed/item_categoria.csv": "fe57e95e8a87f822e2f8f48b0f4b833d14611ca0d53ad59d4d04f022efad4629",
    "data/processed/jogador.csv": "b41ac88e8d557171aba4346e3ef0e6ee857889193ac35690530475dfc1f0e926",
    "data/processed/jogadores_demografia.csv": "f70edb90c79ef271bed7bdb4e6a78a63681daca2d426bbccbad272483a8e302d",
    "data/

## Documento e consultas reais

A amostra mostra apenas a primeira compra; os totais referem-se a todo o documento. O índice de região existe; seu uso depende do plano escolhido pelo otimizador. A migração compara o conteúdo completo de cada coleção temporária antes de substituir o destino. Cada rename é atômico; o conjunto das quatro trocas não é uma transação.

In [3]:
from pymongo import MongoClient
with MongoClient(os.environ["MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
    db = client[os.environ.get("MONGODB_DATABASE", "loja_lol")]
    print(json.dumps(db.jogadores.find_one({"_id": 1}, {"compras": {"$slice": 1}}), ensure_ascii=False, indent=2))
    display(pd.DataFrame(list(db.jogadores.aggregate([
        {"$group": {"_id": "$regiao", "ouro": {"$sum": "$total_gasto_ouro"}, "jogadores": {"$sum": 1}}},
        {"$sort": {"ouro": -1}}
    ]))))
    print("Índices:", list(db.jogadores.index_information()))

{
  "_id": 1,
  "id_jogador": 1,
  "nick": "Faker",
  "id_elo": 10,
  "regiao": "KR",
  "elo": "Desafiante",
  "compras": [
    {
      "id_compra": 72,
      "id_item": 3508,
      "id_partida": 3,
      "data_compra": "2025-08-21",
      "minuto_compra": 18,
      "quantidade": 1,
      "preco_unitario": 3050,
      "item": "Colhedor de Essência",
      "total_ouro": 3050
    }
  ],
  "total_transacoes": 37,
  "total_itens_adquiridos": 41,
  "total_gasto_ouro": 80525
}


,_id,ouro,jogadores
0,BR,1183343,14
1,KR,898220,9
2,EUW,147258,2


Índices: ['_id_', 'regiao_1']


## Análise crítica

Embedding simplifica a consulta de compras de um jogador. Referencing evita duplicar o catálogo e a partida inteira. MongoDB não oferece FKs entre coleções; o validador de aplicação assume essa verificação. `jsonSchema` não substitui uma FK. Não medimos ganho de latência SQL/NoSQL e não configuramos sharding. Em uma carga concorrente, a troca de várias coleções exige coordenação adicional.